# TF-IDF (concatenated) + 1 hidden layer MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:**

- same preprocessing as in the glove notebooks
- `TfidfVectorizer` from sklearn, fitted on all single sentences (O1, O2, H1, H2) of the training data
- the four tf-idf vectors are concatenated in fixed order O1, O2, H1, H2
- MLP: input -> hidden -> ReLU -> 2 outputs (one hidden layer less than `tfidf_bigger_mlp.ipynb`)

In [1]:
%pip install -q datasets lightning nltk python-dotenv scikit-learn scipy wandb


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))  # notebooks/project_1 -> shared pipeline package

import lightning as L
import torch
import torch.nn as nn

from pipeline import tracking
from pipeline.data import DATASET, load_splits
from pipeline.features.tfidf import build_features, fit_vectorizer, vectorizer_summary
from pipeline.loaders import make_loader
from pipeline.preprocessing import tokenize_split
from pipeline.preprocessing import nltk_alnum_lower as preprocessing  # swap the preprocessing variant here
from pipeline.training import evaluate_best, fit, training_summary, weights_dir

True

In [3]:
tracking.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
RUN_NAME = "tfidf-concatenate-small-mlp"  # W&B run + artifact name, weights in models/project_1/tfidf_concatenate_small_mlp/weights

config = {
    "dataset": DATASET,
    **preprocessing.CONFIG,  # tokenizer, lowercase, remove_punctuation, remove_stopwords, normalization, truncation
    "vectorizer": "sklearn.TfidfVectorizer",
    "vocabulary": "train sentences (O1, O2, H1, H2)",
    "example_representation": "concat",  # O1, O2, H1, H2
    "classifier": "mlp",  # linear -> relu -> linear
    "hidden_dim": 256,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}

run = tracking.init_run(RUN_NAME, config)

In [5]:
train, valid, test = load_splits()

train: 16801, valid: 1000, test: 1532


In [8]:
train_tokens = tokenize_split(train, preprocessing.preprocess)
valid_tokens = tokenize_split(valid, preprocessing.preprocess)
test_tokens = tokenize_split(test, preprocessing.preprocess)

In [9]:
vectorizer = fit_vectorizer(train_tokens)

# one tf-idf matrix per field, glued side by side -> (examples, 4 * vocabulary)
X_train, y_train = build_features(train_tokens, vectorizer)
X_valid, y_valid = build_features(valid_tokens, vectorizer)
X_test, y_test = build_features(test_tokens, vectorizer)

print(f"input dimension: {X_train.shape[1]}")

vocabulary: 16189 terms
input dimension: 64756


### TFIDF Vectorizer

### DataLoaders - Takes care of Training Batching and Loading

In [10]:
L.seed_everything(config["seed"])

# sparse tf-idf matrix -> SparseDataset, a row is only densified when the DataLoader asks for it
train_loader = make_loader(X_train, y_train, config["batch_size"], shuffle=True)
valid_loader = make_loader(X_valid, y_valid, config["batch_size"], shuffle=False)
test_loader = make_loader(X_test, y_test, config["batch_size"], shuffle=False)

Seed set to 42


In [11]:
class MLP(L.LightningModule):

    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate):
        super().__init__()
        self.save_hyperparameters()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def reset_parameters(self):
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                layer.reset_parameters()

    def forward(self, x):
        return self.layers(x)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

    def shared_step(self, batch, split):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

### Best Checkpoint + Upload Artifacts to Weights and Biases

In [12]:
model = MLP(
    input_dim=X_train.shape[1],
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    learning_rate=config["learning_rate"],
)
model.reset_parameters()

trainer = fit(model, train_loader, valid_loader, weights_dir(RUN_NAME), max_epochs=config["max_epochs"], patience=config["patience"])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layers  │ Sequential       │ 16.6 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 16.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 16.6 M                                                                                               
Total estimated model params size (MB): 66.313                                                                     
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Metric loss/valid improved. New best score: 0.670
Monitored metric loss/valid did not improve in the last 5 records. Best score: 0.670. Signaling Trainer to stop.


In [13]:
evaluate_best(trainer, model, valid_loader, test_loader)

run.summary.update({
    **training_summary(trainer, y_test),  # accuracy/majority, best/loss_valid, best/checkpoint, early_stopping/stopped_epoch
    **vectorizer_summary(vectorizer, X_train),  # tfidf/vocabulary_size, tfidf/input_dim
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/tfidf_concatenate_small_mlp/weights/best-epoch=0.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/tfidf_concatenate_small_mlp/weights/best-epoch=0.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │    0.5740000009536743     │
│        loss/valid         │    0.6695731282234192     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/tfidf_concatenate_small_mlp/weights/best-epoch=0.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/tfidf_concatenate_small_mlp/weights/best-epoch=0.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.5163185596466064     │
│         loss/test         │    0.7077447175979614     │
└───────────────────────────┴───────────────────────────┘

In [14]:
tracking.log_best_checkpoint(run, RUN_NAME, trainer.checkpoint_callback)
run.finish()

accuracy/test,▁
accuracy/train,▁▅▇███
accuracy/valid,▇█▄█▁▂▇
epoch,▁▁▂▂▃▃▅▅▆▆▇▇██
loss/test,▁
loss/train,█▆▃▁▁▁
loss/valid,▁▂▄▆▇█▁
trainer/global_step,▁▁▂▂▄▄▅▅▇▇████
accuracy/majority,0.50979
accuracy/test,0.51632
accuracy/train,0.99994
